# Inspect alignment interaction tasks

Smoke notebook for scenario 1 from `in_distribution_scenarios.md`: interaction target, min-history filter, random per-user split, and sampled non-interaction candidates.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def _add_repo_root_to_path() -> None:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "pyproject.toml").exists() and (path / "runners").exists():
            for import_path in (str(path), str(path / "src")):
                if import_path not in sys.path:
                    sys.path.insert(0, import_path)
            return
    raise RuntimeError("Could not find beyond-click-sim repo root")


_add_repo_root_to_path()

from runners.in_distribution.interaction_prediction.task_builders import (
    data_root,
    repo_root,
)

REPO_ROOT = repo_root()
CANONICAL_ROOT = data_root()

print("repo:", REPO_ROOT)
print("canonical:", CANONICAL_ROOT)


repo: /Users/a.agafonov/Research/agent_simulator/repos/beyond-click-sim
canonical: /Users/a.agafonov/Research/agent_simulator/data/canonical


In [2]:
from beyond_click_sim.data.canonical import CanonicalDataset
from beyond_click_sim.tasks import (
    AlignmentInteractionTaskBuilder,
    MinUserInteractionsFilter,
    NonInteractionCandidateSampler,
    RandomFractionSplitter,
)


def canonical_dataset(name: str, version: str = "v1") -> CanonicalDataset:
    root = CANONICAL_ROOT / name / version
    return CanonicalDataset(
        name=name,
        version=version,
        root=root,
        users_path=root / "users.parquet",
        items_path=root / "items.parquet",
        interactions_path=root / "interactions.parquet",
        manifest_path=root / "manifest.json",
    )


datasets = {
    "ml-1m": canonical_dataset("ml-1m"),
    "steam": canonical_dataset("steam"),
}

for name, dataset in datasets.items():
    print(name, dataset.root, dataset.interactions_path.exists())

ml-1m /Users/a.agafonov/Research/agent_simulator/data/canonical/ml-1m/v1 True
steam /Users/a.agafonov/Research/agent_simulator/data/canonical/steam/v1 True


In [3]:
MIN_INTERACTIONS = 10
TRAIN_FRACTION = 0.7
VAL_FRACTION = 0.0
TEST_FRACTION = 0.3
NEGATIVE_RATIO = 1
SEED = 0


def make_builder(dataset_name: str) -> AlignmentInteractionTaskBuilder:
    return AlignmentInteractionTaskBuilder(
        name=f"{dataset_name}-interaction-alignment-1to{NEGATIVE_RATIO}",
        dataset_filter=MinUserInteractionsFilter(min_interactions=MIN_INTERACTIONS),
        splitter=RandomFractionSplitter(
            train_fraction=TRAIN_FRACTION,
            val_fraction=VAL_FRACTION,
            test_fraction=TEST_FRACTION,
            seed=SEED,
            group_column="user_id",
            stratify_column=None,
        ),
        sampler=NonInteractionCandidateSampler(
            negative_ratio=NEGATIVE_RATIO,
            seed=SEED,
        ),
    )

In [4]:
def split_summary(task) -> pd.DataFrame:
    rows = []
    target_col = task.schema.target_column
    sampled_col = task.schema.sampled_column
    for split_name in ["train", "val", "test"]:
        frame = getattr(task, split_name)
        rows.append(
            {
                "split": split_name,
                "rows": len(frame),
                "users": frame["user_id"].nunique() if len(frame) else 0,
                "items": frame["item_id"].nunique() if len(frame) else 0,
                "target_mean": frame[target_col].mean() if len(frame) else pd.NA,
                "sampled_rows": int(frame[sampled_col].sum()) if sampled_col and len(frame) else 0,
            }
        )
    return pd.DataFrame(rows)


def candidate_group_summary(frame: pd.DataFrame, group_col: str, target_col: str, sampled_col: str) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(
            [{"candidate_sets": 0, "rows_min": 0, "rows_max": 0, "positives_min": 0, "positives_max": 0}]
        )
    grouped = frame.groupby(group_col, sort=False).agg(
        rows=(target_col, "size"),
        positives=(target_col, "sum"),
        sampled=(sampled_col, "sum"),
    )
    return pd.DataFrame(
        [
            {
                "candidate_sets": len(grouped),
                "rows_min": int(grouped["rows"].min()),
                "rows_max": int(grouped["rows"].max()),
                "positives_min": int(grouped["positives"].min()),
                "positives_max": int(grouped["positives"].max()),
                "sampled_min": int(grouped["sampled"].min()),
                "sampled_max": int(grouped["sampled"].max()),
            }
        ]
    )


def leakage_summary(task, dataset: CanonicalDataset) -> pd.DataFrame:
    sampled_col = task.schema.sampled_column
    observed = dataset.load_interactions(columns=["user_id", "item_id"])
    observed = observed.drop_duplicates().assign(_observed=1)
    rows = []
    for split_name in ["val", "test"]:
        frame = getattr(task, split_name)
        negatives = frame.loc[frame[sampled_col].astype(bool), ["user_id", "item_id"]]
        checked = negatives.merge(observed, on=["user_id", "item_id"], how="left")
        rows.append(
            {
                "split": split_name,
                "sampled_negatives": len(negatives),
                "observed_pair_leaks": int(checked["_observed"].notna().sum()),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
tasks = {}

for dataset_name, dataset in datasets.items():
    print("=" * 100)
    print(dataset_name)
    builder = make_builder(dataset_name)
    task = builder.build(dataset)
    tasks[dataset_name] = task

    manifest_view = {
        key: task.manifest[key]
        for key in [
            "dataset",
            "dataset_version",
            "target_source_column",
            "target_column",
            "filter",
            "splitter",
            "sampler",
            "rows",
            "users",
            "items",
        ]
    }
    print(json.dumps(manifest_view, indent=2, ensure_ascii=False))

    display(split_summary(task))
    display(
        candidate_group_summary(
            task.test,
            task.schema.candidate_group_column,
            task.schema.target_column,
            task.schema.sampled_column,
        )
    )
    display(leakage_summary(task, dataset))

ml-1m


In [ ]:
for dataset_name, task in tasks.items():
    print("=" * 100)
    print(dataset_name, "schema")
    print(task.schema)
    print("feature columns:", len(task.schema.feature_columns))
    print(task.schema.feature_columns)

    if not task.test.empty:
        first_group = task.test[task.schema.candidate_group_column].iloc[0]
        print("example candidate_group:", first_group)
        display(task.test[task.test[task.schema.candidate_group_column].eq(first_group)].head(10))